# 01 — Drift Compensation Loop

**Repository:** `quantum-hardware-company`  
**Directory:** `control-stack/`

This notebook starts the control-stack layer by turning calibration drift estimates into a simple closed-loop compensation workflow.

## Purpose

Calibration measured that Rabi frequency and readout offset drift over repeated blocks.

This notebook asks:

> If we can estimate drift, can we compensate the next control setting?

## Workflow

1. Generate synthetic target hardware behavior.
2. Simulate uncompensated drift.
3. Estimate drift from calibration blocks.
4. Apply a control update.
5. Compare uncompensated vs compensated response.
6. Track CGCS / Triplet-Phase-Lock stability.
7. Save figures and a machine-readable JSON summary.

## Principle

Control software follows measured physics.

The control stack should not assume ideal pulses.  
It should adapt from calibration evidence.


## Control model

For Rabi-like calibration, the observed response depends on:

$$
P(t)=A\sin^2\left(\frac{\Omega t}{2}\right)+B
$$

Drift changes effective parameters:

$$
\Omega_k = \Omega_{target}+\delta\Omega_k
$$

$$
B_k = B_{target}+\delta B_k
$$

A simple compensation rule is:

$$
u_{\Omega,k}=\Omega_{target}-\hat{\delta\Omega}_k
$$

$$
u_{B,k}=B_{target}-\hat{\delta B}_k
$$

The compensated effective parameters become closer to target when estimates are accurate.


In [ ]:
import os
import json

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

np.random.seed(9423)

FIG_DIR = "figures"
RESULTS_DIR = "results"

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)


In [ ]:
def rabi_model(t, A, Omega, B):
    """Rabi oscillation probability model."""
    return A * np.sin(0.5 * Omega * t) ** 2 + B


def fit_model(model, x, y, p0, bounds=(-np.inf, np.inf)):
    """Fit model to data and return best-fit params + covariance."""
    params, cov = curve_fit(model, x, y, p0=p0, bounds=bounds, maxfev=30000)
    return params, cov


def phase_lock_metric(signal, fit):
    """Cosine similarity between observed signal and target/model signal."""
    dot = np.dot(signal, fit)
    norm = np.linalg.norm(signal) * np.linalg.norm(fit)
    if norm == 0:
        return np.nan
    return dot / norm


def is_phase_locked(metric, threshold=1 / np.sqrt(2)):
    """CGCS threshold: cos(theta) >= 1/sqrt(2), equivalent to <= 45 degrees."""
    return metric >= threshold


def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))


def moving_average(x, window=3):
    """Causal moving average estimate."""
    y = np.zeros_like(x, dtype=float)
    for i in range(len(x)):
        lo = max(0, i - window + 1)
        y[i] = np.mean(x[lo:i + 1])
    return y


## Generate target behavior and uncompensated drift

This simulates a hardware system whose effective Rabi frequency and readout offset drift over lab time.


In [ ]:
# Calibration blocks / lab-time index
n_blocks = 36
block_times = np.arange(n_blocks)

# Pulse duration axis inside each calibration block
n_points_per_block = 140
pulse_t = np.linspace(0, 10, n_points_per_block)

# Target parameters
target_A = 0.88
target_Omega = 2.45
target_B = 0.06

target_signal = rabi_model(pulse_t, target_A, target_Omega, target_B)

# True environmental drift
true_delta_Omega = (
    0.055 * np.sin(2 * np.pi * block_times / 21 + 0.4)
    + 0.018 * np.sin(2 * np.pi * block_times / 9)
)
true_delta_B = (
    0.0016 * block_times
    + 0.006 * np.sin(2 * np.pi * block_times / 14 + 0.2)
)

# Uncompensated effective parameters
Omega_uncomp = target_Omega + true_delta_Omega
B_uncomp = target_B + true_delta_B
A_uncomp = target_A + 0.012 * np.sin(2 * np.pi * block_times / 17)

print(f"Target Omega = {target_Omega:.4f}")
print(f"Target B     = {target_B:.4f}")
print(f"Uncompensated Omega range = {Omega_uncomp.min():.4f} to {Omega_uncomp.max():.4f}")
print(f"Uncompensated B range     = {B_uncomp.min():.4f} to {B_uncomp.max():.4f}")


## Simulate calibration measurements

Each block is measured with noise. We then fit the block to estimate its effective parameters.


In [ ]:
Y_uncomp_obs = []
Y_uncomp_clean = []

for k in range(n_blocks):
    y_clean = rabi_model(pulse_t, A_uncomp[k], Omega_uncomp[k], B_uncomp[k])
    noise = 0.04 * np.random.randn(n_points_per_block)
    y_obs = np.clip(y_clean + noise, 0, 1)
    Y_uncomp_clean.append(y_clean)
    Y_uncomp_obs.append(y_obs)

Y_uncomp_clean = np.array(Y_uncomp_clean)
Y_uncomp_obs = np.array(Y_uncomp_obs)

fit_params = []
fit_stds = []
fit_curves = []

initial_guess = [0.85, 2.40, 0.05]
bounds = ([0.0, 0.0, 0.0], [1.2, 5.0, 0.3])

for k in range(n_blocks):
    params, cov = fit_model(
        rabi_model,
        pulse_t,
        Y_uncomp_obs[k],
        p0=initial_guess,
        bounds=bounds,
    )
    fit_params.append(params)
    fit_stds.append(np.sqrt(np.diag(cov)))
    fit_curves.append(rabi_model(pulse_t, *params))

fit_params = np.array(fit_params)
fit_stds = np.array(fit_stds)
fit_curves = np.array(fit_curves)

A_est = fit_params[:, 0]
Omega_est = fit_params[:, 1]
B_est = fit_params[:, 2]

delta_Omega_est = Omega_est - target_Omega
delta_B_est = B_est - target_B

print("Uncompensated calibration fit complete.")
print(f"Mean estimated Omega drift = {np.mean(delta_Omega_est):.6f}")
print(f"Mean estimated B drift     = {np.mean(delta_B_est):.6f}")


## Estimate drift for control

A real control loop should avoid overreacting to single noisy calibration points.

Here we use a simple causal moving average. Later this can be replaced by a Kalman filter, AR model, or learned predictor.


In [ ]:
window = 4

delta_Omega_hat = moving_average(delta_Omega_est, window=window)
delta_B_hat = moving_average(delta_B_est, window=window)

# Control commands attempt to cancel estimated drift.
Omega_command = target_Omega - delta_Omega_hat
B_command = target_B - delta_B_hat

# Residual drift after compensation.
# For this synthetic model, commanded values subtract from the environmental drift.
Omega_comp = target_Omega + true_delta_Omega - delta_Omega_hat
B_comp = target_B + true_delta_B - delta_B_hat

# Keep offset physically bounded for simulated readout.
B_comp = np.clip(B_comp, 0, 0.3)

print(f"Control estimator window = {window}")
print(f"Compensated Omega range = {Omega_comp.min():.4f} to {Omega_comp.max():.4f}")
print(f"Compensated B range     = {B_comp.min():.4f} to {B_comp.max():.4f}")


## Simulate compensated response

This compares the original uncompensated response to the compensated response produced by the control update.


In [ ]:
Y_comp_clean = []
Y_comp_obs = []

for k in range(n_blocks):
    y_clean = rabi_model(pulse_t, target_A, Omega_comp[k], B_comp[k])
    noise = 0.04 * np.random.randn(n_points_per_block)
    y_obs = np.clip(y_clean + noise, 0, 1)
    Y_comp_clean.append(y_clean)
    Y_comp_obs.append(y_obs)

Y_comp_clean = np.array(Y_comp_clean)
Y_comp_obs = np.array(Y_comp_obs)

print("Compensated response simulated.")


## Drift estimates and compensation commands


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(block_times, true_delta_Omega, linewidth=2, label="true Omega drift")
plt.plot(block_times, delta_Omega_est, marker="o", linewidth=1, label="estimated Omega drift")
plt.plot(block_times, delta_Omega_hat, linewidth=2, linestyle="--", label="control estimate")
plt.axhline(0, linewidth=1)
plt.xlabel("calibration block / lab time index")
plt.ylabel("Omega drift")
plt.title("Control-stack drift estimate: Rabi frequency")
plt.legend()
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/control_01_omega_drift_estimate.png", dpi=300)
plt.savefig(f"{FIG_DIR}/control_01_omega_drift_estimate.pdf")

plt.show()


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(block_times, true_delta_B, linewidth=2, label="true offset drift")
plt.plot(block_times, delta_B_est, marker="o", linewidth=1, label="estimated offset drift")
plt.plot(block_times, delta_B_hat, linewidth=2, linestyle="--", label="control estimate")
plt.axhline(0, linewidth=1)
plt.xlabel("calibration block / lab time index")
plt.ylabel("readout offset drift")
plt.title("Control-stack drift estimate: readout offset")
plt.legend()
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/control_02_offset_drift_estimate.png", dpi=300)
plt.savefig(f"{FIG_DIR}/control_02_offset_drift_estimate.pdf")

plt.show()


## Uncompensated vs compensated parameter error

This is the core control-stack result.

A useful compensation loop should reduce parameter error relative to target.


In [ ]:
Omega_error_uncomp = Omega_uncomp - target_Omega
Omega_error_comp = Omega_comp - target_Omega

B_error_uncomp = B_uncomp - target_B
B_error_comp = B_comp - target_B

plt.figure(figsize=(9, 5))
plt.plot(block_times, Omega_error_uncomp, marker="o", linewidth=1, label="uncompensated Omega error")
plt.plot(block_times, Omega_error_comp, marker="o", linewidth=1, label="compensated Omega error")
plt.axhline(0, linewidth=1)
plt.xlabel("calibration block / lab time index")
plt.ylabel("Omega error from target")
plt.title("Drift compensation: Rabi frequency error reduction")
plt.legend()
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/control_03_omega_error_reduction.png", dpi=300)
plt.savefig(f"{FIG_DIR}/control_03_omega_error_reduction.pdf")

plt.show()


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(block_times, B_error_uncomp, marker="o", linewidth=1, label="uncompensated offset error")
plt.plot(block_times, B_error_comp, marker="o", linewidth=1, label="compensated offset error")
plt.axhline(0, linewidth=1)
plt.xlabel("calibration block / lab time index")
plt.ylabel("offset error from target")
plt.title("Drift compensation: readout offset error reduction")
plt.legend()
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/control_04_offset_error_reduction.png", dpi=300)
plt.savefig(f"{FIG_DIR}/control_04_offset_error_reduction.pdf")

plt.show()


## Response-level improvement

Parameter control matters because it makes measured responses stay closer to target behavior.


In [ ]:
uncomp_response_rmse = np.array([rmse(Y_uncomp_clean[k], target_signal) for k in range(n_blocks)])
comp_response_rmse = np.array([rmse(Y_comp_clean[k], target_signal) for k in range(n_blocks)])

plt.figure(figsize=(9, 5))
plt.plot(block_times, uncomp_response_rmse, marker="o", linewidth=1, label="uncompensated response RMSE")
plt.plot(block_times, comp_response_rmse, marker="o", linewidth=1, label="compensated response RMSE")
plt.xlabel("calibration block / lab time index")
plt.ylabel("RMSE vs target response")
plt.title("Drift compensation: response-level error reduction")
plt.legend()
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/control_05_response_error_reduction.png", dpi=300)
plt.savefig(f"{FIG_DIR}/control_05_response_error_reduction.pdf")

plt.show()


## Example block comparison

Visual comparison for one late block where uncompensated drift is visible.


In [ ]:
example_block = int(np.argmax(uncomp_response_rmse))

plt.figure(figsize=(9, 5))
plt.plot(pulse_t, target_signal, linewidth=2, label="target response")
plt.plot(pulse_t, Y_uncomp_clean[example_block], linewidth=2, label="uncompensated response")
plt.plot(pulse_t, Y_comp_clean[example_block], linewidth=2, label="compensated response")
plt.xlabel("time / pulse duration")
plt.ylabel("excited-state probability")
plt.title(f"Control comparison for block {example_block}")
plt.legend()
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/control_06_example_block_comparison.png", dpi=300)
plt.savefig(f"{FIG_DIR}/control_06_example_block_comparison.pdf")

plt.show()


## CGCS stability before and after compensation

We compare each block against the target response.

This measures whether compensation keeps the system closer to the constraint region defined by the target behavior.


In [ ]:
threshold = 1 / np.sqrt(2)

metric_uncomp = np.array([
    phase_lock_metric(Y_uncomp_clean[k], target_signal) for k in range(n_blocks)
])

metric_comp = np.array([
    phase_lock_metric(Y_comp_clean[k], target_signal) for k in range(n_blocks)
])

angle_uncomp = np.degrees(np.arccos(np.clip(metric_uncomp, -1, 1)))
angle_comp = np.degrees(np.arccos(np.clip(metric_comp, -1, 1)))

plt.figure(figsize=(9, 5))
plt.axhline(threshold, linewidth=1, linestyle="--", label="45° threshold")
plt.plot(block_times, metric_uncomp, marker="o", linewidth=1, label="uncompensated")
plt.plot(block_times, metric_comp, marker="o", linewidth=1, label="compensated")
plt.ylim(0.85, 1.01)
plt.xlabel("calibration block / lab time index")
plt.ylabel("cosine similarity vs target")
plt.title("CGCS stability: uncompensated vs compensated")
plt.legend()
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/control_07_cgcs_stability_comparison.png", dpi=300)
plt.savefig(f"{FIG_DIR}/control_07_cgcs_stability_comparison.pdf")

plt.show()


## Compensation summary metrics


In [ ]:
Omega_rmse_uncomp = rmse(Omega_uncomp, np.full(n_blocks, target_Omega))
Omega_rmse_comp = rmse(Omega_comp, np.full(n_blocks, target_Omega))

B_rmse_uncomp = rmse(B_uncomp, np.full(n_blocks, target_B))
B_rmse_comp = rmse(B_comp, np.full(n_blocks, target_B))

response_rmse_uncomp_mean = float(np.mean(uncomp_response_rmse))
response_rmse_comp_mean = float(np.mean(comp_response_rmse))

Omega_improvement = 1 - Omega_rmse_comp / Omega_rmse_uncomp
B_improvement = 1 - B_rmse_comp / B_rmse_uncomp
response_improvement = 1 - response_rmse_comp_mean / response_rmse_uncomp_mean

summary = {
    "n_blocks": int(n_blocks),
    "estimator_window": int(window),
    "Omega_rmse_uncompensated": float(Omega_rmse_uncomp),
    "Omega_rmse_compensated": float(Omega_rmse_comp),
    "Omega_error_reduction_fraction": float(Omega_improvement),
    "B_rmse_uncompensated": float(B_rmse_uncomp),
    "B_rmse_compensated": float(B_rmse_comp),
    "B_error_reduction_fraction": float(B_improvement),
    "response_rmse_uncompensated_mean": float(response_rmse_uncomp_mean),
    "response_rmse_compensated_mean": float(response_rmse_comp_mean),
    "response_error_reduction_fraction": float(response_improvement),
    "min_phase_lock_uncompensated": float(np.min(metric_uncomp)),
    "min_phase_lock_compensated": float(np.min(metric_comp)),
    "mean_phase_lock_uncompensated": float(np.mean(metric_uncomp)),
    "mean_phase_lock_compensated": float(np.mean(metric_comp)),
    "max_angle_uncompensated_degrees": float(np.max(angle_uncomp)),
    "max_angle_compensated_degrees": float(np.max(angle_comp)),
    "phase_lock_threshold": float(threshold),
    "all_compensated_blocks_phase_locked": bool(np.all(metric_comp >= threshold)),
}

summary


## Save control-stack summary


In [ ]:
with open(f"{RESULTS_DIR}/drift_compensation_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"Saved {RESULTS_DIR}/drift_compensation_summary.json")


## Zip outputs for download from Colab

Run this cell after all figures/results have been generated.


In [ ]:
!zip -r drift_compensation_outputs.zip figures results


## Control-stack status

This notebook creates the first closed-loop bridge:

```text
calibration/drift/ → control-stack/
```

It turns measured drift into control updates.

## Next steps

Possible next notebooks:

- `02_closed_loop_pulse_update.ipynb` — update pulse duration / amplitude from calibration estimates.
- `03_kalman_drift_filter.ipynb` — replace moving average with a state-space drift filter.
- `04_control_policy_comparison.ipynb` — compare no control, moving average, proportional control, and predictive control.
- `noise-mitigation/01_structured_drift_model.ipynb` — model residual drift as a noise process.

Guiding rule:

> Control follows calibration evidence.
